In [1]:
import time
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = "https://wos-help.webofscience.com/WOKRS535R111/help/WOS/{key}_abrvjt.html"
PAGE_KEYS = ["0-9"] + list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/123.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

def is_abbrev_line(raw_line: str) -> bool:
    """
    经验规则：缩写行通常缩进更深（常见含制表符\t或更大的前导空白）。
    """
    if "\t" in raw_line:
        return True
    leading_ws = len(raw_line) - len(raw_line.lstrip(" \t"))
    return leading_ws >= 8

def parse_one_page(html: str):
    soup = BeautifulSoup(html, "lxml")

    # 用纯文本解析（页面里“Journal List”之后就是正文列表）
    text = soup.get_text("\n")
    lines = text.splitlines()

    # 找到列表起点
    start_idx = None
    for i, ln in enumerate(lines):
        if ln.strip() == "Journal List":
            start_idx = i
            break
    if start_idx is None:
        raise RuntimeError("Cannot find 'Journal List' in page text.")

    entries = []
    current_title = None
    has_abbrev = False

    for raw in lines[start_idx + 1:]:
        raw = raw.rstrip("\r\n")
        s = raw.strip()
        if not s:
            continue

        if is_abbrev_line(raw) and current_title:
            # 这是缩写（页面上为粗体）
            entries.append({"full_title": current_title, "abbrev": s})
            has_abbrev = True
        else:
            # 这是一个新的全称
            if current_title and not has_abbrev:
                entries.append({"full_title": current_title, "abbrev": "NA"})
            current_title = s
            has_abbrev = False

    # 收尾：最后一个全称如果没缩写，补 NA
    if current_title and not has_abbrev:
        entries.append({"full_title": current_title, "abbrev": "NA"})

    return entries

def fetch(url: str, session: requests.Session, retries: int = 3, sleep_s: float = 1.0) -> str:
    last_err = None
    for _ in range(retries):
        try:
            r = session.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            # 该站一般是 utf-8，如遇乱码可改成 r.encoding = r.apparent_encoding
            r.encoding = "utf-8"
            return r.text
        except Exception as e:
            last_err = e
            time.sleep(sleep_s)
    raise RuntimeError(f"Failed to fetch {url}: {last_err}")

def main():
    session = requests.Session()

    all_rows = []
    for key in PAGE_KEYS:
        url = BASE_URL.format(key=key)
        print(f"Fetching {key}: {url}")
        html = fetch(url, session=session, retries=3, sleep_s=1.2)

        rows = parse_one_page(html)
        for row in rows:
            row["page_key"] = key
            row["source_url"] = url
        all_rows.extend(rows)

        # 礼貌一点，避免给站点压力过大
        time.sleep(0.8)

    df_raw = pd.DataFrame(all_rows)

    # 去重版本：同一(全称,缩写)只保留一条
    df_unique = df_raw.drop_duplicates(subset=["full_title", "abbrev"]).reset_index(drop=True)

    # 输出
    df_raw.to_csv("wos_journal_abbrev_raw.csv", index=False, encoding="utf-8-sig")
    df_unique.to_csv("wos_journal_abbrev_unique.csv", index=False, encoding="utf-8-sig")

    with pd.ExcelWriter("wos_journal_abbrev.xlsx", engine="openpyxl") as writer:
        df_unique.to_excel(writer, sheet_name="unique", index=False)
        df_raw.to_excel(writer, sheet_name="raw", index=False)

    print("Done!")
    print("Saved:")
    print(" - wos_journal_abbrev.xlsx (sheets: unique/raw)")
    print(" - wos_journal_abbrev_unique.csv")
    print(" - wos_journal_abbrev_raw.csv")

if __name__ == "__main__":
    main()


Fetching 0-9: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/0-9_abrvjt.html
Fetching A: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/A_abrvjt.html
Fetching B: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/B_abrvjt.html
Fetching C: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/C_abrvjt.html
Fetching D: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/D_abrvjt.html
Fetching E: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/E_abrvjt.html
Fetching F: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/F_abrvjt.html
Fetching G: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/G_abrvjt.html
Fetching H: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/H_abrvjt.html
Fetching I: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/I_abrvjt.html
Fetching J: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/J_abrvjt.html
Fetching K: https://wos-help.webofscience.com/WOKRS535R111/help/WOS/K_abrvjt.html
Fetching L: 